# U2 AquaVigia: ingesta USGS y procesamiento streaming (Spark)

Dimension U2 de Celia Patricia Apaza Hilasaca: temperatura y oxigeno disuelto en tiempo real.
Fuente: sensores continuos reales del USGS (EE. UU.), publicados en Kafka por `usgs_productor.py`.
La infraestructura (Kafka, topico, productor) es compartida con Jhenderson Machaca (pH y turbidez); las variables, el modelo, el evento de negocio y los paneles son propios.

Este cuaderno cubre los criterios 2 (ingesta IoT verificada) y 3 (ventana, watermark y checkpoint sobre datos reales).

In [1]:
import os, json, time, shutil, math
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType

INTEGRANTE = "Patricia Apaza"
VARIABLES = ["temperatura_c", "oxigeno_mgl"]
KAFKA = "kafka:9092"
TOPIC = "agua-usgs-eventos"
BASE = "/opt/s12-u2-usgs"
ARTIFACTS = f"{BASE}/artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("u2-usgs-streaming")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "1")
    .config("spark.sql.session.timeZone", "America/Lima")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b744c141-dbc4-4d82-9654-994fb5d78980;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

## 1. Contrato del evento (esquema explicito)

| Campo | Tipo | Significado |
|---|---|---|
| evento_id | string | identificador unico del evento |
| estacion_id | string | estacion USGS, p. ej. USGS-01594441 |
| variable | string | ph, oxigeno_mgl, turbidez_fnu, conductividad_uscm, temperatura_c, nitrato_mgl |
| codigo_parametro | string | codigo USGS (00010 = temperatura, 00300 = oxigeno, ...) |
| valor | double | lectura del sensor |
| fecha_medicion_utc | string ISO-8601 | tiempo del EVENTO (cuando el sensor midio) |
| fecha_ingesta_utc | string ISO-8601 | cuando llego a nuestro pipeline |
| retraso_min | double | minutos entre medicion e ingesta |
| origen | string | vivo, historial o replay |
| fuente | string | USGS |

El tiempo del evento es `fecha_medicion_utc`, no la hora de llegada: es la base del watermark.

In [2]:
esquema_evento = (
    StructType()
    .add("evento_id", StringType())
    .add("estacion_id", StringType())
    .add("variable", StringType())
    .add("codigo_parametro", StringType())
    .add("valor", DoubleType())
    .add("fecha_medicion_utc", StringType())
    .add("fecha_ingesta_utc", StringType())
    .add("retraso_min", DoubleType())
    .add("origen", StringType())
    .add("fuente", StringType())
)

def leer_kafka_batch(inicio="earliest"):
    return (spark.read.format("kafka")
            .option("kafka.bootstrap.servers", KAFKA).option("subscribe", TOPIC)
            .option("startingOffsets", inicio).option("endingOffsets", "latest").load())

def leer_kafka_stream(inicio="latest", max_por_lote=None):
    r = (spark.readStream.format("kafka")
         .option("kafka.bootstrap.servers", KAFKA).option("subscribe", TOPIC)
         .option("startingOffsets", inicio).option("failOnDataLoss", "false"))
    if max_por_lote:
        r = r.option("maxOffsetsPerTrigger", max_por_lote)
    return r.load()

def publicar_en_kafka(registros):
    """registros: lista de (key, value_json). Publica en el mismo topico con el productor de Spark."""
    df = spark.createDataFrame(registros, ["key", "value"])
    (df.write.format("kafka").option("kafka.bootstrap.servers", KAFKA).option("topic", TOPIC).save())

def parsear(df_kafka, filtrar=True):
    """value (binario) -> columnas tipadas + ts (tiempo de EVENTO). Descarta lo que no cumple el contrato.
    filtrar=True deja solo las variables de la dimension propia (VARIABLES)."""
    d = (
        df_kafka.selectExpr("CAST(value AS STRING) AS json_str")
        .select(F.from_json(F.col("json_str"), esquema_evento).alias("e")).select("e.*")
        .withColumn("ts", F.to_timestamp("fecha_medicion_utc"))
        .withColumn("ts_ingesta", F.to_timestamp("fecha_ingesta_utc"))
        .where(F.col("estacion_id").isNotNull() & F.col("variable").isNotNull()
               & F.col("valor").isNotNull() & F.col("ts").isNotNull())
    )
    return d.where(F.col("variable").isin(VARIABLES)) if filtrar else d

## 2. Ingesta verificada: lo que hay en el topico (criterio 2)

In [3]:
lote = leer_kafka_batch("earliest")
n_total = lote.count()
print("mensajes en el topico:", n_total)
lote.selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value", "partition", "offset") \
    .orderBy("offset", ascending=False).show(3, truncate=80)

todos = parsear(lote, filtrar=False).cache()
validos = todos.where(F.col("variable").isin(VARIABLES)).cache()
n_validos = validos.count()
n_todos = todos.count()
print(f"eventos validos segun el contrato: {n_todos} de {n_total}  ({n_todos / max(n_total,1):.1%})")
print(f"de ellos, de la dimension de {INTEGRANTE} {VARIABLES}: {n_validos}")
print("eventos unicos (estacion, variable, tiempo):",
      validos.dropDuplicates(["estacion_id", "variable", "fecha_medicion_utc"]).count())

mensajes en el topico: 6072


+-------------+--------------------------------------------------------------------------------+---------+------+
|          key|                                                                           value|partition|offset|
+-------------+--------------------------------------------------------------------------------+---------+------+
|USGS-01649190|{"evento_id": "e7bea093-6196-4655-aa10-c69fd379bdb6", "estacion_id": "USGS-01...|        0|  6071|
|USGS-01491000|{"evento_id": "c153b903-5c6d-41c4-bbb4-eaec4aa52d97", "estacion_id": "USGS-01...|        0|  6070|
|USGS-01491000|{"evento_id": "0445994b-86a8-4342-b3ed-7667b48876c3", "estacion_id": "USGS-01...|        0|  6069|
+-------------+--------------------------------------------------------------------------------+---------+------+
only showing top 3 rows


eventos validos segun el contrato: 6072 de 6072  (100.0%)
de ellos, de la dimension de Patricia Apaza ['temperatura_c', 'oxigeno_mgl']: 2323
eventos unicos (estacion, variable, tiempo): 2323


In [4]:
print("Por origen:")
validos.groupBy("origen").count().show()
print("Por variable (todas las del topico):")
todos.groupBy("variable").agg(F.count("*").alias("eventos"), F.countDistinct("estacion_id").alias("estaciones"),
                                F.round(F.avg("valor"), 2).alias("promedio")).orderBy("variable").show()
print(f"Estaciones con las variables de {INTEGRANTE}:")
(validos.groupBy("estacion_id").pivot("variable").count().orderBy("estacion_id").show(30, truncate=False))

Por origen:
+------+-----+
|origen|count|
+------+-----+
|  vivo| 2323|
+------+-----+

Por variable (todas las del topico):
+------------------+-------+----------+--------+
|          variable|eventos|estaciones|promedio|
+------------------+-------+----------+--------+
|conductividad_uscm|   1161|        12|  362.34|
|       nitrato_mgl|    361|         7|    1.12|
|       oxigeno_mgl|   1162|        12|    8.88|
|                ph|   1161|        12|     7.6|
|     temperatura_c|   1161|        12|   14.37|
|      turbidez_fnu|   1066|        11|    2.08|
+------------------+-------+----------+--------+

Estaciones con las variables de Patricia Apaza:
+---------------+-----------+-------------+
|estacion_id    |oxigeno_mgl|temperatura_c|
+---------------+-----------+-------------+
|USGS-01491000  |101        |101          |
|USGS-01579550  |96         |96           |
|USGS-01594441  |96         |96           |
|USGS-01640152  |96         |96           |
|USGS-0164578734|99         

## 3. Retraso real de los datos y eleccion del watermark

Los sensores miden cada 5 minutos pero USGS los publica en lotes. El `retraso_min` de los eventos en vivo mide cuanto tarda un dato en llegar. El watermark debe ser mayor que ese retraso o se descartarian datos legitimos.

In [5]:
DESDE = "2026-10-30T06:08:00"
LIMPIO_HASTA = "2026-10-30T06:42:00"
vivo_todo = validos.where((F.col("origen") == "vivo") & (F.col("fecha_ingesta_utc") >= DESDE))
vivo_limpio = vivo_todo.where(F.col("fecha_ingesta_utc") <= LIMPIO_HASTA)

def cuantiles(d, nombre):
    n = d.count()
    if n == 0:
        print(f"{nombre}: sin eventos"); return n, None
    q = d.approxQuantile("retraso_min", [0.5, 0.9, 0.99, 1.0], 0.001)
    print(f"{nombre}: n={n}  mediana={q[0]:.0f}  p90={q[1]:.0f}  p99={q[2]:.0f}  maximo={q[3]:.0f} min")
    return n, q

n_t, q_t = cuantiles(vivo_todo, "Todos los vivos (incluye cortes mios)")
n_l, q_l = cuantiles(vivo_limpio, "Ventana limpia (sin cortes)       ")
if q_l is not None and n_l >= 100:
    q, base = q_l, "ventana limpia"
elif q_t is not None:
    q, base = q_t, "todos los vivos"
else:
    q, base = None, "ninguno"
if q is not None:
    WM_MIN = int(max(60, math.ceil(q[2] * 1.2 / 5) * 5))
else:
    print("Aun no hay eventos en vivo: se usa 60 min (retraso de 40 a 54 min medido el 29-oct-2026).")
    WM_MIN = 60
print(f"WATERMARK elegido: {WM_MIN} minutos  (regla: max(60, p99 x 1.2 redondeado a 5), base: {base})")
VENTANA = "15 minutes"
WATERMARK = f"{WM_MIN} minutes"


Todos los vivos (incluye cortes mios): n=745  mediana=34  p90=68  p99=148  maximo=168 min
Ventana limpia (sin cortes)       : n=144  mediana=28  p90=56  p99=71  maximo=71 min
WATERMARK elegido: 85 minutos  (regla: max(60, p99 x 1.2 redondeado a 5), base: ventana limpia)


## 4. Streaming: leer, parsear y agregar por ventana

Ventana de 15 minutos (tres lecturas de 5 minutos), por estacion y variable. Se usa `startingOffsets=earliest` y `maxOffsetsPerTrigger` para que el stream recorra el topico en varios micro-lotes y se vea el avance del watermark.

In [6]:
def agregar(df, ancho=VENTANA, paso=None):
    w = F.window("ts", ancho, paso) if paso else F.window("ts", ancho)
    return (df.groupBy(w, "estacion_id", "variable")
              .agg(F.count("*").alias("lecturas"), F.round(F.avg("valor"), 3).alias("promedio"),
                   F.round(F.min("valor"), 3).alias("minimo"), F.round(F.max("valor"), 3).alias("maximo"))
              .select(F.col("window.start").alias("inicio"), F.col("window.end").alias("fin"),
                      F.col("window.start").cast("long").alias("inicio_epoch"),
                      "estacion_id", "variable", "lecturas", "promedio", "minimo", "maximo"))

consultas = []
def chk_nuevo(nombre):
    ruta = f"/tmp/u2_chk_{nombre}_{int(time.time())}"
    shutil.rmtree(ruta, ignore_errors=True)
    return ruta

def correr(df, modo, trigger="5 seconds", checkpoint=None, nombre="q"):
    """Arranca una consulta y guarda en `lotes` cada micro-lote: (batchId, filas, columnas)."""
    lotes = []
    def al_lote(d, bid):
        lotes.append((bid, [list(r) for r in d.limit(5000).collect()], d.columns))
    q = (df.writeStream.outputMode(modo).foreachBatch(al_lote)
         .option("checkpointLocation", checkpoint or chk_nuevo(nombre))
         .trigger(processingTime=trigger).queryName(nombre).start())
    consultas.append(q)
    return q, lotes

def esperar(seg):
    time.sleep(seg)

def detener_todo():
    for q in spark.streams.active:
        q.stop()
    print("consultas activas:", len(spark.streams.active))

def mostrar_lotes(lotes, ultimos=2):
    for bid, filas, cols in lotes[-ultimos:]:
        print(f"--- micro-lote {bid}: {len(filas)} filas (hasta 5000)")
        print(pd.DataFrame(filas, columns=cols).head(6).to_string(index=False))

crudo = leer_kafka_stream("earliest", max_por_lote=3000)
eventos = parsear(crudo)
print("es un stream:", eventos.isStreaming)
eventos.printSchema()

es un stream: True
root
 |-- evento_id: string (nullable = true)
 |-- estacion_id: string (nullable = true)
 |-- variable: string (nullable = true)
 |-- codigo_parametro: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- fecha_medicion_utc: string (nullable = true)
 |-- fecha_ingesta_utc: string (nullable = true)
 |-- retraso_min: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- fuente: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- ts_ingesta: timestamp (nullable = true)



### 4.1 Sin watermark: el estado crece sin limite

In [7]:
sin_wm = agregar(eventos)
q1, lotes_sin = correr(sin_wm, "complete", trigger="3 seconds", nombre="sin_watermark")
esperar(40)
q1.stop()
filas_por_lote = [(b, len(f)) for b, f, _ in lotes_sin]
print("filas de salida por micro-lote (modo complete):", filas_por_lote[-8:])
print("El resultado crece con cada lote: Spark conserva TODAS las ventanas.")

filas de salida por micro-lote (modo complete): [(0, 405), (1, 790), (2, 798)]
El resultado crece con cada lote: Spark conserva TODAS las ventanas.


### 4.2 Con watermark y checkpoint: el estado se acota (criterio 3)

In [8]:
CHK_WM = "/tmp/u2_chk_watermark"
shutil.rmtree(CHK_WM, ignore_errors=True)
con_wm = agregar(eventos.withWatermark("ts", WATERMARK))
q2, lotes_wm = correr(con_wm, "update", trigger="3 seconds", checkpoint=CHK_WM, nombre="con_watermark")
esperar(40)
q2.stop()
print("micro-lotes:", len(lotes_wm), " batchId:", [b for b, _, _ in lotes_wm])
mostrar_lotes(lotes_wm, ultimos=1)

micro-lotes: 4  batchId: [0, 1, 2, 3]
--- micro-lote 3: 0 filas (hasta 5000)
Empty DataFrame
Columns: [inicio, fin, inicio_epoch, estacion_id, variable, lecturas, promedio, minimo, maximo]
Index: []


In [9]:
print("Contenido del checkpoint:", sorted(c for c in os.listdir(CHK_WM)))
print("offsets/:", sorted(f for f in os.listdir(f"{CHK_WM}/offsets") if not f.startswith("."))[:12])
print("commits/:", sorted(f for f in os.listdir(f"{CHK_WM}/commits") if not f.startswith("."))[:12])
print("state/  :", sorted(os.listdir(f"{CHK_WM}/state")))
ultimo = sorted(f for f in os.listdir(f"{CHK_WM}/offsets") if not f.startswith("."))[-1]
print(f"\n--- offsets/{ultimo} ---")
print(open(f"{CHK_WM}/offsets/{ultimo}").read()[:400])

Contenido del checkpoint: ['.metadata.crc', 'commits', 'metadata', 'offsets', 'sources', 'state']
offsets/: ['0', '1', '2', '3']
commits/: ['0', '1', '2', '3']
state/  : ['0']

--- offsets/3 ---
v1
{"batchWatermarkMs":1793343900000,"batchTimestampMs":1793353584007,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider","spark.sql.streaming.stateStore.rocksdb.mergeOperatorVersion":"2","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.queryEvolution.enableSourceEvolution":"false","spar


### 4.3 Reinicio con el MISMO checkpoint: los batchId continuan, no vuelven a 0

In [10]:
# Para VER que la numeracion continua, se reenvian 60 lecturas reales recientes (duplicados, solo para generar trabajo; se declara).
q3, lotes_reinicio = correr(con_wm, "update", trigger="3 seconds", checkpoint=CHK_WM, nombre="reinicio")
esperar(8)
ult = (lote.selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value", "offset")
       .orderBy(F.desc("offset")).limit(60).collect())
publicar_en_kafka([(r["key"], r["value"]) for r in ult])
esperar(20)
q3.stop()
print("batchId de la corrida original :", [b for b, _, _ in lotes_wm])
print("batchId tras reiniciar         :", [b for b, _, _ in lotes_reinicio])
print("Si tras reiniciar los batchId siguen la numeracion (y no vuelven a 0), el checkpoint funciona.")


batchId de la corrida original : [0, 1, 2, 3]
batchId tras reiniciar         : [4]
Si tras reiniciar los batchId siguen la numeracion (y no vuelven a 0), el checkpoint funciona.


### 4.4 Dato tardio: con watermark se descarta, sin watermark reaparece

In [11]:
raw = lote.selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value", "offset")
mias = (raw.withColumn("e", F.from_json("value", esquema_evento))
        .where(F.col("e.variable").isin(VARIABLES)).withColumn("t", F.to_timestamp("e.fecha_medicion_utc")))
antigua = mias.orderBy("t").limit(1).collect()[0]
recientes = mias.orderBy(F.desc("t")).limit(40).collect()
j = json.loads(antigua["value"])
t_max = recientes[0]["t"]
brecha_min = (t_max - antigua["t"]).total_seconds() / 60
EPOCH_VENTANA = int(pd.Timestamp(j["fecha_medicion_utc"]).floor("15min").timestamp())
print("lectura antigua que se reenvia:", j["estacion_id"], j["variable"], j["fecha_medicion_utc"])
print("ventana que le toca (epoch)  :", EPOCH_VENTANA)
print(f"brecha con la lectura mas reciente: {brecha_min:.0f} min   watermark: {WM_MIN} min  ->",
      "llega TARDE (valido para la prueba)" if brecha_min > WM_MIN else "NO llega tarde: la prueba no vale")


[Stage 66:>                                                         (0 + 1) / 1]

lectura antigua que se reenvia: USGS-01491000 temperatura_c 2026-10-30T00:10:00+00:00
ventana que le toca (epoch)  : 1793318400
brecha con la lectura mas reciente: 500 min   watermark: 85 min  -> llega TARDE (valido para la prueba)


In [12]:
def ev_stream():
    return parsear(leer_kafka_stream("latest"))

qA, lotes_A = correr(agregar(ev_stream().withWatermark("ts", WATERMARK)), "update", trigger="4 seconds", nombre="tardio_con")
qB, lotes_B = correr(agregar(ev_stream()), "update", trigger="4 seconds", nombre="tardio_sin")
esperar(12)
publicar_en_kafka([(r["key"], r["value"]) for r in recientes])   # adelanta el reloj de eventos (duplicados de lecturas reales)
esperar(14)
publicar_en_kafka([(r["key"], r["value"]) for r in recientes[:5]])   # otro lote: el watermark se aplica al lote siguiente
esperar(14)
marca_A, marca_B = len(lotes_A), len(lotes_B)
publicar_en_kafka([(antigua["key"], antigua["value"])])
print(">>> lectura antigua publicada de nuevo (llega tarde)")
esperar(20)
qA.stop(); qB.stop()

def aparece(lotes, desde):
    for _, filas, cols in lotes[desde:]:
        k = cols.index("inicio_epoch")
        if any(r[k] == EPOCH_VENTANA and r[cols.index("estacion_id")] == j["estacion_id"]
               and r[cols.index("variable")] == j["variable"] for r in filas):
            return True
    return False

print("Ventana del dato tardio CON watermark:", aparece(lotes_A, marca_A), " (esperado: False, se descarta)")
print("Ventana del dato tardio SIN watermark:", aparece(lotes_B, marca_B), " (esperado: True, reaparece)")


>>> lectura antigua publicada de nuevo (llega tarde)
Ventana del dato tardio CON watermark: False  (esperado: False, se descarta)
Ventana del dato tardio SIN watermark: True  (esperado: True, reaparece)


### 4.5 Salida persistente (Parquet) con checkpoint

In [13]:
PARQUET_OUT = f"{ARTIFACTS}/usgs_agregado_parquet"
CHK_PARQUET = "/tmp/u2_chk_parquet"
shutil.rmtree(PARQUET_OUT, ignore_errors=True); shutil.rmtree(CHK_PARQUET, ignore_errors=True)

def guardar(d, bid):
    if d.count() == 0:
        return
    d.withColumn("batchId", F.lit(bid)).write.mode("append").parquet(PARQUET_OUT)

q4 = (agregar(eventos.withWatermark("ts", WATERMARK)).writeStream.outputMode("update")
      .foreachBatch(guardar).option("checkpointLocation", CHK_PARQUET)
      .trigger(processingTime="3 seconds").start())
esperar(40)
q4.stop()
res = spark.read.parquet(PARQUET_OUT)
print("filas de ventanas guardadas:", res.count())
res.where(F.col("variable").isin(VARIABLES)).orderBy(F.desc("inicio")).show(8, truncate=False)

filas de ventanas guardadas: 830
+-------------------+-------------------+------------+---------------+-------------+--------+--------+------+------+-------+
|inicio             |fin                |inicio_epoch|estacion_id    |variable     |lecturas|promedio|minimo|maximo|batchId|
+-------------------+-------------------+------------+---------------+-------------+--------+--------+------+------+-------+
|2026-10-30 03:30:00|2026-10-30 03:45:00|1793349000  |USGS-01491000  |temperatura_c|6       |13.0    |13.0  |13.0  |2      |
|2026-10-30 03:30:00|2026-10-30 03:45:00|1793349000  |USGS-01491000  |oxigeno_mgl  |6       |8.6     |8.6   |8.6   |2      |
|2026-10-30 03:15:00|2026-10-30 03:30:00|1793348100  |USGS-0164579522|temperatura_c|6       |13.733  |13.7  |13.8  |2      |
|2026-10-30 03:15:00|2026-10-30 03:30:00|1793348100  |USGS-01491000  |oxigeno_mgl  |12      |8.533   |8.5   |8.6   |2      |
|2026-10-30 03:15:00|2026-10-30 03:30:00|1793348100  |USGS-0164579522|oxigeno_mgl  |6       

In [14]:
detener_todo()

consultas activas: 0
